# ProcessBehavior Tutorial: Fill Weight Analysis

This notebook demonstrates **iterative process behavior analysis** using fill weight data from a 4-lane filling system.

## What We'll Cover

### Part 1: Start Simple - Lane Analysis
- Load and explore data
- Analyze by lane only
- Understand Xbar/S charts and appropriate WECO rules
- Detect signals correctly

### Part 2: Go Deeper - Lane + Phase Analysis  
- Add fill cycle phase (up/down arm position)
- Full variance decomposition with residuals
- Main effects and interactions
- Export comprehensive results

## The Data

**Filling System**: 4 lanes, each with up/down fill arm motion
- **pull**: Production sequence (1-100 time points)
- **lane**: Filling lane (1-4)
- **phase**: Fill cycle position (1=down, 2=up)
- **fill_weight**: Target measurement

---

# Part 1: Start Simple - Analyze by Lane

## Step 1: Load and Explore Data

In [ ]:
# Import required libraries
import pandas as pd
from processbehavior import ProcessDataFrame
from processbehavior.signals import SignalConfig
from pathlib import Path

# Find the data file (works from any directory)
data_paths = [
    'processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from project root
    '../../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from examples/tom/
]

df = None
for path in data_paths:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"✓ Loaded data from: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "Could not find FILLWEIGHTDATA_800.csv. "
        "Make sure you're running this notebook from the project root or examples/tom/ directory."
    )

# Explore the data
print(f"\nDataset: {len(df)} observations")
print(f"  • Pulls (time points): {df['pull'].nunique()}")
print(f"  • Lanes: {sorted(df['lane'].unique())}")
print(f"  • Phases: {sorted(df['phase'].unique())}")
print(f"  • Missing values: {df['fill_weight'].isna().sum()}")

df.head(12)  # Show first 12 rows (3 pulls × 4 lanes)

## Step 2: Create ProcessDataFrame

The `ProcessDataFrame` automatically handles missing values and garbage characters in your data.

In [ ]:
# Create ProcessDataFrame - automatically cleans data
pdf = ProcessDataFrame(df)

print(f"ProcessDataFrame ready with {len(pdf.data)} observations")
print(f"  • Cleaned/removed: {len(df) - len(pdf.data)} rows with missing values")

## Step 3: Analyze by Lane Only

**Question**: Are the 4 lanes performing consistently over time?

We'll group by `lane` only, treating phase as replicates within each lane/pull combination.

In [ ]:
# Run lane-only analysis
analysis_lane = pdf.analyze(
    response_var=pdf.columns.fill_weight,
    grouping_vars=[pdf.columns.lane],  # Group by lane only
    time_var=pdf.columns.pull          # Time sequence
)

result_lane = analysis_lane.calculate()

print("\n" + "=" * 80)
print("LANE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {result_lane.summary['sds']} - {result_lane.summary['sds_description']}")
print(f"Analysis: {result_lane.summary['analysis_type']}")
print(f"Charts Available: {list(result_lane.charts.keys())}")
print(f"Observations: {result_lane.summary['n_observations']}")

### Understanding the Results

**SDS 3**: Partial Replication
- Most (lane × pull) cells have 2 measurements (phase 1 & 2)
- Some cells have missing values → mixed replication
- System uses hybrid variance estimation

**Available Charts**:
- **Xbar**: Average fill weight by lane over time
- **Sbar**: Variation within each lane over time

**Important**: These charts show **categorical comparisons** (lane 1 vs lane 2 vs lane 3 vs lane 4), not individual sequential measurements over time.

## Step 4: Visualize Lane Performance

In [ ]:
# Plot Xbar and S charts
fig_lane = result_lane.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_lane.show()

**Interpretation Tips**:
- **Xbar Chart**: Shows if lane averages are stable over time
- **Sbar Chart**: Shows if lane variation is consistent
- Points beyond limits indicate special causes
- **Interactive**: Hover over points to see details!

## Step 5: Understanding WECO Rules for Xbar/S Charts

### Important Distinction: Chart Types and Rule Applicability

**Xbar and S charts** show categorical comparisons of rational subgroups:
- Each point = mean (or stddev) of a subgroup
- Points represent different categories (lanes), not sequential individual measurements
- Consecutive points on the chart are NOT in temporal sequence

**WECO Rules Applicability**:
- ✅ **Rule 1** (Point beyond control limits): Valid for all chart types
- ❌ **Rules 2-4** (Sequential patterns): Only valid for time series charts (IMR, R)
  - Rule 2: "2 of 3 consecutive" - needs temporal sequence
  - Rule 3: "4 of 5 consecutive" - needs temporal sequence  
  - Rule 4: "8+ run" - needs temporal sequence

**Why this matters**: Applying sequential rules to categorical comparisons produces meaningless results. The lanes are not "consecutive" in any temporal sense.

**For time-series analysis**: Use stratified IMR charts (shown in Part 2).

## Step 6: Detect Signals (Correct Usage)

In [ ]:
# Detect signals using only Rule 1 (appropriate for Xbar/S charts)
signals_xbar_lane = result_lane.detect_signals(
    chart='Xbar',
    rules='default',  # Uses chart-type-based defaults (Rule 1 only for Xbar)
    config=SignalConfig(min_observations=2)  # Lowered for tutorial
)

signals_sbar_lane = result_lane.detect_signals(
    chart='Sbar',
    rules='default',  # Uses chart-type-based defaults (Rule 1 only for S)
    config=SignalConfig(min_observations=2)
)

print("Signal Detection - Lane Analysis:")
print(f"  Xbar (means): {signals_xbar_lane.count} signals (Rule 1 only)")
print(f"  Sbar (variation): {signals_sbar_lane.count} signals (Rule 1 only)")
print("\nNote: Only Rule 1 applied - appropriate for categorical comparisons")

if signals_xbar_lane.has_signals:
    print(f"\nXbar Signals:")
    print(signals_xbar_lane.violations[['rule_name', 'value', 'description']].head(10))

## Step 7: Check Lane Main Effects

Are some lanes systematically different from others?

In [ ]:
if result_lane.has_effects and 'lane' in result_lane.effects:
    print("Lane Main Effects:")
    print(result_lane.effects['lane'])
    print("\nInterpretation:")
    print("  • 'Main_Effect' shows deviation from grand mean")
    print("  • Positive = higher than average")
    print("  • Negative = lower than average")
else:
    print("Main effects not available for this configuration")

---

# Part 2: Go Deeper - Add Phase Analysis

## Step 8: Analyze Lane + Phase

**Question**: Does the fill arm position (up/down) affect fill weight differently across lanes?

Now we'll include **both lane and phase** as grouping variables to see the full picture.

In [ ]:
# Run lane + phase analysis
analysis_full = pdf.analyze(
    response_var=pdf.columns.fill_weight,
    grouping_vars=[pdf.columns.lane, pdf.columns.phase],  # Both factors
    time_var=pdf.columns.pull
)

result_full = analysis_full.calculate()

print("\n" + "=" * 80)
print("LANE + PHASE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {result_full.summary['sds']} - {result_full.summary['sds_description']}")
print(f"Analysis: {result_full.summary['analysis_type']}")
print(f"Charts Available: {list(result_full.charts.keys())}")
print(f"Observations: {result_full.summary['n_observations']}")
print(f"\nCapabilities:")
print(f"  • Residuals: {result_full.has_residuals}")
print(f"  • Main Effects: {result_full.has_effects}")
print(f"  • Interactions: {result_full.has_interactions}")

### Understanding the Change

**SDS 2**: No Replication
- Each (lane × phase × pull) cell has exactly 1 measurement
- No true replicates (phase is now a factor, not a replicate)
- Uses moving range for variance estimation
- Full main effects and interaction analysis available

## Step 9: Visualize Full Analysis

In [ ]:
# Plot all available charts
fig_full = result_full.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_full.show()

## Step 10: Variance Decomposition (Residuals)

**Wheeler's Residuals** (R1-R5) break down variation into components:
- **R1**: Total deviation from grand mean
- **R2**: Time effects removed
- **R3**: Factor (lane/phase) effects removed
- **R4**: Both time and factor effects removed
- **R5**: Pure error (all systematic effects removed)

In [ ]:
if result_full.has_residuals:
    print("Residuals Available: Yes\n")
    
    # Show first 10 observations with residuals
    display_cols = ['pull', 'lane', 'phase', 'fill_weight', 'R1', 'R2', 'R3', 'R4', 'R5']
    print("Sample Data with Residuals:")
    print(result_full.dataset[display_cols].head(10))
    
    # Calculate variance by residual type
    print("\n\nVariance Breakdown:")
    for col in ['R1', 'R2', 'R3', 'R4', 'R5']:
        var = result_full.residuals[col].var()
        print(f"  {col}: {var:.2f}")
else:
    print("Residuals not available for this SDS")

## Step 11: Main Effects Analysis

### Lane and Phase Effects
Do lanes differ systematically? Does phase matter?

In [ ]:
if result_full.has_effects:
    if 'lane' in result_full.effects:
        print("Lane Main Effects:")
        print(result_full.effects['lane'])
        print()
    
    if 'phase' in result_full.effects:
        print("\nPhase Main Effects (Up vs Down):")
        print(result_full.effects['phase'])
        print("\nInterpretation:")
        print("  • Phase 1 (down) vs Phase 2 (up) comparison")
        print("  • Shows if arm position affects fill weight")

## Step 12: Interaction Effects

Do lanes behave differently in up vs down positions?

In [ ]:
if result_full.has_interactions:
    print("Interaction Analysis Available: Yes\n")
    
    # Show average by lane-phase combination
    interaction_summary = result_full.dataset.groupby(['lane', 'phase'])['fill_weight'].agg(['mean', 'count'])
    print("Lane × Phase Combinations:")
    print(interaction_summary)
    print("\nInterpretation:")
    print("  • Each cell shows average fill weight for that lane/phase combo")
    print("  • Look for patterns: do some lanes change more between phases?")
else:
    print("Interactions not available for this configuration")

## Step 13: Signal Detection (Full Analysis)

In [ ]:
# Detect signals in full analysis
signals_full = result_full.detect_signals(
    chart='Xbar',
    rules='default',  # Only Rule 1 for Xbar (categorical comparisons)
    config=SignalConfig(min_observations=2)
)

print(f"Signal Detection - Full Analysis:")
print(f"  • Signals Found: {signals_full.has_signals}")
print(f"  • Total Count: {signals_full.count}")
print(f"  • Rules Applied: Rule 1 only (appropriate for Xbar charts)")

if signals_full.has_signals:
    print(f"\nSignal Details (first 15):")
    print(signals_full.violations[['rule_name', 'value', 'description']].head(15))

## Step 14: Time Series Analysis with Stratified IMR

**Want to use Rules 2-4?** Use stratified IMR charts!

IMR charts show individual measurements over time - these ARE true time series where sequential rules apply.

In [ ]:
# Create stratified analysis (separate IMR chart for each lane)
analysis_stratified = pdf.analyze(
    response_var=pdf.columns.fill_weight,
    grouping_vars=[pdf.columns.lane, pdf.columns.phase],
    time_var=pdf.columns.pull,
    stratify=True  # Creates separate IMR charts per lane-phase combination
)

result_stratified = analysis_stratified.calculate()

print("Stratified Analysis:")
print(f"  Charts created: {len(result_stratified.charts)}")
print(f"  Chart names: {list(result_stratified.charts.keys())[:8]}...")  # Show first 8

### Detect Signals on IMR Charts (All Rules Apply)

In [ ]:
# Detect signals on a specific IMR chart
# Example: Lane 1, Phase 1
imr_chart_name = [k for k in result_stratified.charts.keys() if k.startswith('Imr')][0]

print(f"\nAnalyzing: {imr_chart_name}")

signals_imr = result_stratified.detect_signals(
    chart=imr_chart_name,
    rules='standard',  # Rules 1-4 are valid for IMR (time series!)
    config=SignalConfig(min_observations=10)
)

print(f"  • Signals Found: {signals_imr.has_signals}")
print(f"  • Total Count: {signals_imr.count}")
print(f"  • Rules Applied: 1-4 (appropriate for IMR time series)")

if signals_imr.has_signals:
    print(f"\nIMR Signal Details:")
    print(signals_imr.violations[['rule_name', 'value', 'description']].head(10))

## Step 15: Export Comprehensive Results

Save everything to Excel for sharing with the team or further analysis.

In [ ]:
# Export full analysis to Excel
result_full.to_excel(
    'fillweight_analysis_complete.xlsx',
    include_residuals=True,
    include_effects=True,
    include_interactions=True,
    include_full_dataset=True
)

print("✅ Results exported to: fillweight_analysis_complete.xlsx")
print("\nWorkbook includes:")
print("  • Summary sheet with analysis metadata")
print("  • Xbar chart data (subgroup means)")
print("  • Sbar chart data (subgroup variation)")
print("  • VAS Residuals (R1-R5 with original columns)")
print("  • Main Effects (lane and phase)")
print("  • Interactions (lane × phase)")
print("  • Full Dataset (all calculated values)")

---

# Summary: Iterative Analysis Workflow

## What We Learned

### Stage 1: Lane-Only Analysis (SDS 3)
✓ Quick overview of lane performance  
✓ Identify which lanes have issues  
✓ Detect signals using Rule 1 (appropriate for Xbar/S)  
✓ Phase treated as replicates  

### Stage 2: Lane + Phase Analysis (SDS 2)
✓ Understand effect of fill arm position  
✓ Full variance decomposition (R1-R5)  
✓ Separate main effects for lane and phase  
✓ Interaction analysis (lane × phase)  
✓ Comprehensive Excel export  

### Stage 3: Stratified IMR (Optional)
✓ True time series analysis per lane-phase combination  
✓ All WECO rules (1-4) applicable  
✓ Detect trends, runs, and patterns over time  

## Key Insights

**Chart Types Matter**:
- **Xbar/S**: Categorical comparisons → Only Rule 1 applies
- **IMR/R**: Time series → All rules (1-8) apply
- Understanding the data structure determines which rules make sense

**Iterative Analysis is Powerful**:
1. Start simple (fewer grouping variables)
2. Identify areas needing investigation
3. Add complexity (more grouping variables)
4. Use stratification for time-series drill-down

**ProcessBehavior Adapts**:
- Automatically detects data structure (SDS)
- Recommends appropriate charts
- Applies correct WECO rules by default
- Handles missing values gracefully

## Next Steps

### Explore Further
1. **Custom Rule Sets**: Explicitly specify which rules to use
2. **Time Windows**: Analyze specific time ranges for process changes
3. **Multiple Responses**: Analyze other measurements (weight, volume, etc.)
4. **Compare Strategies**: Use different grouping strategies for different insights

### Production Use
- Use Wheeler's recommended minimum: 20+ observations for signal detection
- Document special causes when signals occur
- Recalculate limits after process changes
- Share Excel reports with the team

---

**Questions?** This workflow demonstrates how real practitioners use process behavior charts - understanding when different chart types and rules apply is key to correct analysis!